# TomTom Traffic — Vector Flow Tiles (PBF)

This notebook explores the TomTom Traffic Flow API and builds a pipeline to enrich
OSM road edges with `congestion_tomtom` using the same map-matching logic as Mapbox.

## How TomTom differs from Mapbox and Google

| Aspect | Mapbox | Google | TomTom |
|---|---|---|---|
| Format | Vector MVT | PNG screenshot (Playwright) | Vector PBF |
| Congestion | Text property (`"heavy"`) | Pixel color | Speed ratio (`currentSpeed / freeFlowSpeed`) |
| Map matching | Spatial join needed | Bearing-based pixel matching | Spatial join needed |
| API key | Mapbox token | Google Maps JS API key | TomTom API key |
| Free tier | Yes | Yes (Maps JS API) | 2,500 tile requests/day |

## Congestion from speed ratio

| Speed ratio | Level |
|---|---|
| > 0.85 | `low` |
| 0.60 – 0.85 | `moderate` |
| 0.40 – 0.60 | `heavy` |
| < 0.40 | `severe` |

## Pipeline

```
Step 1 — Inspect one tile (discover layer names + properties)
Step 2 — Download all boundary tiles
Step 3 — Decode PBF → compute speed ratio → congestion level
Step 4 — Save traffic GeoJSON
Step 5 — Map match to OSM edges → write congestion_tomtom to DuckDB
Step 6 — Visualize + compare with Mapbox and Google
```

---
## Configuration

Add `TOMTOM_API_KEY=...` to `../.env` before running.
Free key at [developer.tomtom.com](https://developer.tomtom.com) — 2,500 tile requests/day.

In [ ]:
%%time
import os, sys, yaml, io, time
import numpy as np
import requests
import mercantile
import geopandas as gpd
import pandas as pd
import folium
import matplotlib.pyplot as plt
from pathlib import Path
from shapely.geometry import box, shape, LineString
from shapely.ops import transform as shapely_transform
import pyproj
import mapbox_vector_tile
from shapely.affinity import affine_transform
from concurrent.futures import ThreadPoolExecutor, as_completed
from dotenv import load_dotenv

sys.path.insert(0, '..')
from traffic_db import TrafficDB, CONGESTION_COLORS, CONGESTION_ORDER

load_dotenv(Path('../.env'), override=True)

# ── Configuration ─────────────────────────────────────────────────────────
NAME       = 'sodermalm'
ZOOM       = 14      # zoom 14 = same as Mapbox default (~9.5 m/tile pixel)
OUTPUT_DIR = Path('../output')
TILES_DIR  = Path('../tiles/tomtom')
# ─────────────────────────────────────────────────────────────────────────

TOMTOM_API_KEY = os.environ.get('TOMTOM_API_KEY', '')
if not TOMTOM_API_KEY:
    raise ValueError("TOMTOM_API_KEY not set in ../.env")
print(f'TomTom API key : {TOMTOM_API_KEY[:8]}...{TOMTOM_API_KEY[-4:]}')

CONFIG_PATH = Path(f'../config/{NAME}.yaml')
if CONFIG_PATH.exists():
    with open(CONFIG_PATH) as f:
        cfg = yaml.safe_load(f)
    DB_PATH       = Path(cfg['output_path']) / f"{cfg['name']}.duckdb"
    BOUNDARY_FILE = Path(cfg['boundary_path'])
else:
    DB_PATH       = Path(f'../db/{NAME}.duckdb')
    BOUNDARY_FILE = Path(f'../boundaries/{NAME}.geojson')

TILES_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

print(f'DuckDB    : {DB_PATH}')
print(f'Boundary  : {BOUNDARY_FILE}')
print(f'Zoom      : {ZOOM}')

---
## Step 1 — Inspect one tile

Download a single PBF tile covering Södermalm and print everything decoded from it.
This reveals the exact **layer name** and **property names** we need to use.

In [ ]:
%%time
# Sample tile covering central Södermalm at zoom 14
SAMPLE_TILE = mercantile.tile(18.065, 59.315, ZOOM)
print(f'Sample tile: z={SAMPLE_TILE.z}  x={SAMPLE_TILE.x}  y={SAMPLE_TILE.y}')

TOMTOM_FLOW_URL = (
    f"https://api.tomtom.com/traffic/map/4/tile/flow/relative"
    f"/{SAMPLE_TILE.z}/{SAMPLE_TILE.x}/{SAMPLE_TILE.y}.pbf"
)

r = requests.get(TOMTOM_FLOW_URL, params={'key': TOMTOM_API_KEY}, timeout=15)
print(f'HTTP {r.status_code}  |  {len(r.content):,} bytes')

if r.status_code == 200:
    data = mapbox_vector_tile.decode(r.content)
    print(f'\nLayer names in tile: {list(data.keys())}')

    for layer_name, layer in data.items():
        features = layer.get('features', [])
        print(f'\n── Layer: "{layer_name}"  ({len(features)} features) ──')
        if features:
            # Show properties of first feature
            f0 = features[0]
            print(f'  Geometry type : {f0["geometry"]["type"]}')
            print(f'  Properties    : {f0["properties"]}')
            # Show range of speed values
            speeds = [f['properties'].get('currentSpeed', None) for f in features if f['properties'].get('currentSpeed') is not None]
            ff_speeds = [f['properties'].get('freeFlowSpeed', None) for f in features if f['properties'].get('freeFlowSpeed') is not None]
            if speeds:
                print(f'  currentSpeed  : min={min(speeds)}  max={max(speeds)}  ({len(speeds)} features with data)')
            if ff_speeds:
                print(f'  freeFlowSpeed : min={min(ff_speeds)}  max={max(ff_speeds)}')
else:
    print(f'ERROR: {r.text[:200]}')

---
## Step 2 — Download all boundary tiles

Now that we know the format, download all tiles covering the boundary in parallel.

In [ ]:
%%time
gdf      = gpd.read_file(BOUNDARY_FILE).to_crs('EPSG:4326')
boundary = gdf.geometry.union_all()
w, s, e, n = boundary.bounds

tiles = [
    t for t in mercantile.tiles(w, s, e, n, zooms=ZOOM)
    if boundary.intersects(box(*mercantile.bounds(t)))
]
print(f'Boundary  : W={w:.4f} S={s:.4f} E={e:.4f} N={n:.4f}')
print(f'Tiles     : {len(tiles)} at zoom {ZOOM}')

In [ ]:
%%time
def download_tomtom_tile(tile, out_dir, api_key, retries=3):
    path = out_dir / f"{tile.z}_{tile.x}_{tile.y}.pbf"
    if path.exists() and path.stat().st_size > 0:
        return tile, path
    url = (f"https://api.tomtom.com/traffic/map/4/tile/flow/relative"
           f"/{tile.z}/{tile.x}/{tile.y}.pbf")
    for attempt in range(retries):
        try:
            r = requests.get(url, params={'key': api_key}, timeout=(5, 20))
        except requests.exceptions.Timeout:
            time.sleep(2 ** attempt); continue
        if r.status_code == 200:
            path.write_bytes(r.content)
            return tile, path
        if r.status_code == 429:
            time.sleep(2 ** attempt)
        else:
            print(f'  {tile} HTTP {r.status_code}')
            return tile, None
    return tile, None

tile_paths = {}
failed     = []
with ThreadPoolExecutor(max_workers=6) as pool:
    futs = {pool.submit(download_tomtom_tile, t, TILES_DIR, TOMTOM_API_KEY): t for t in tiles}
    for f in as_completed(futs):
        tile, path = f.result()
        if path:
            tile_paths[tile] = path
        else:
            failed.append(tile)

print(f'Downloaded : {len(tile_paths)}/{len(tiles)} tiles')
if failed:
    print(f'Failed     : {len(failed)}')

---
## Step 3 — Decode PBF and compute congestion levels

Decode each tile, compute `currentSpeed / freeFlowSpeed` for each road segment,
and classify into `low` / `moderate` / `heavy` / `severe`.

Also reproject from MVT pixel coordinates to EPSG:4326 using the same affine
transform approach as the Mapbox pipeline.

In [ ]:
%%time
# ── Update these after running Step 1 if the names differ ────────────────
TOMTOM_LAYER = 'Traffic flow'      # layer name inside the PBF
PROP_CURRENT  = 'currentSpeed'     # current speed property
PROP_FREEFLOW = 'freeFlowSpeed'    # free-flow speed property
# ─────────────────────────────────────────────────────────────────────────

SPEED_THRESHOLDS = [
    ('severe',   0.40),
    ('heavy',    0.60),
    ('moderate', 0.85),
]

def speed_ratio_to_congestion(current, free_flow):
    if free_flow <= 0:
        return 'no data'
    ratio = current / free_flow
    for level, threshold in SPEED_THRESHOLDS:
        if ratio < threshold:
            return level
    return 'low'

def decode_tomtom_tile(path, tile):
    """Decode one TomTom PBF tile → list of (geometry_4326, congestion) tuples."""
    data = mapbox_vector_tile.decode(path.read_bytes())
    layer = data.get(TOMTOM_LAYER, {})
    features = layer.get('features', [])
    if not features:
        return []

    # Affine transform: MVT pixel coords → EPSG:4326
    bounds = mercantile.bounds(tile)
    extent = layer.get('extent', 4096)
    dx = (bounds.east  - bounds.west)  / extent
    dy = (bounds.north - bounds.south) / extent
    # [a, b, d, e, xoff, yoff] → x' = a*x + b*y + xoff, y' = d*x + e*y + yoff
    matrix = [dx, 0, 0, -dy, bounds.west, bounds.north]

    results = []
    for feat in features:
        props = feat.get('properties', {})
        current   = props.get(PROP_CURRENT,  None)
        free_flow = props.get(PROP_FREEFLOW, None)
        if current is None or free_flow is None:
            continue
        congestion = speed_ratio_to_congestion(current, free_flow)

        geom_raw = shape(feat['geometry'])
        geom_4326 = affine_transform(geom_raw, matrix)
        results.append({
            'geometry':    geom_4326,
            'congestion':  congestion,
            'current_kph': round(current, 1),
            'freeflow_kph': round(free_flow, 1),
            'speed_ratio': round(current / max(free_flow, 1), 3),
        })
    return results

# Decode all tiles
all_rows = []
for tile, path in tile_paths.items():
    try:
        all_rows.extend(decode_tomtom_tile(path, tile))
    except Exception as ex:
        print(f'  Could not decode {path.name}: {ex}')

if not all_rows:
    raise RuntimeError('No features decoded — check TOMTOM_LAYER name in Step 1 output')

traffic = gpd.GeoDataFrame(all_rows, crs='EPSG:4326').clip(boundary)

print(f'Decoded    : {len(traffic):,} road segments')
print(f'\nCongestion distribution:')
print(traffic['congestion'].value_counts().to_string())
print(f'\nSpeed ratio range: {traffic["speed_ratio"].min():.2f} – {traffic["speed_ratio"].max():.2f}')

---
## Step 3b — Visualize raw TomTom segments

Show the decoded TomTom road segments colored by congestion before map-matching to OSM edges.

In [ ]:
%%time
lines = traffic[traffic.geometry.geom_type.isin(['LineString', 'MultiLineString'])]
bds    = lines.total_bounds
center = [(bds[1]+bds[3])/2, (bds[0]+bds[2])/2]

m = folium.Map(location=center, zoom_start=14, tiles='OpenStreetMap')
folium.GeoJson(
    gdf.__geo_interface__,
    style_function=lambda _: {'color': 'navy', 'weight': 2, 'fillOpacity': 0.05},
    tooltip=None, popup=None,
).add_to(m)
folium.GeoJson(
    lines.__geo_interface__,
    style_function=lambda feat: {
        'color':  CONGESTION_COLORS.get(feat['properties'].get('congestion', 'no data'), '#cccccc'),
        'weight': 3,
    },
    tooltip=folium.GeoJsonTooltip(
        fields=['congestion', 'current_kph', 'freeflow_kph', 'speed_ratio'],
        aliases=['Congestion', 'Current km/h', 'Free-flow km/h', 'Speed ratio'],
    ),
    popup=None,
).add_to(m)
m

---
## Step 4 — Save traffic GeoJSON

In [ ]:
%%time
traffic_file = OUTPUT_DIR / f'{NAME}_traffic_tomtom.geojson'
traffic.to_file(traffic_file, driver='GeoJSON')
print(f'Saved: {traffic_file}  ({len(traffic):,} segments)')

---
## Step 5 — Map match to OSM edges

Spatially join TomTom road segments to OSM driving edges using the same geometric
map-matching algorithm as the Mapbox pipeline:
- 25 m buffer corridor
- 45° direction filter
- 40% overlap threshold

In [ ]:
%%time
import duckdb
from shapely import get_coordinates

# Load OSM edges
con = duckdb.connect(str(DB_PATH), read_only=True)
con.execute('LOAD spatial')
df = con.execute("""
    SELECT edge_id, osm_id, highway, name, length_m,
           ST_AsText(geometry) AS wkt_geom
    FROM driving.edges
""").df()
con.close()

edges = gpd.GeoDataFrame(
    df,
    geometry=gpd.GeoSeries.from_wkt(df['wkt_geom']),
    crs='EPSG:4326',
).reset_index(drop=True)
edges = edges[edges.geometry.geom_type.isin(['LineString', 'MultiLineString'])]
print(f'OSM edges loaded: {len(edges):,}')

# ── Reuse pipeline map-matching logic ────────────────────────────────────
# Project to EPSG:3857 for metric operations
proj = pyproj.Transformer.from_crs('EPSG:4326', 'EPSG:3857', always_xy=True).transform

SEVERITY = {'severe': 4, 'heavy': 3, 'moderate': 2, 'low': 1}

def bearing(geom):
    coords = get_coordinates(geom)
    if len(coords) < 2:
        return 0.0
    dx = coords[-1][0] - coords[0][0]
    dy = coords[-1][1] - coords[0][1]
    return float(np.degrees(np.arctan2(dy, dx)) % 360)

def dir_diff(b1, b2):
    d = abs(b1 - b2) % 360
    return min(d, 360 - d)

# Buffer edges and build spatial index
edges_m   = edges.to_crs('EPSG:3857')
traffic_m = traffic.to_crs('EPSG:3857')
sindex    = edges_m.sindex

BUF_M          = 25    # metres corridor
DIR_THRESH     = 45    # degrees
OVERLAP_THRESH = 0.40  # fraction

edge_cong = {}
n_matched = 0

for _, seg in traffic_m.iterrows():
    cong = seg['congestion']
    seg_geom = seg.geometry
    seg_buf  = seg_geom.buffer(BUF_M)
    seg_bear = bearing(seg_geom)
    seg_len  = seg_geom.length

    for idx in sindex.intersection(seg_buf.bounds):
        edge    = edges_m.iloc[idx]
        edge_id = int(edge['edge_id'])
        if dir_diff(seg_bear, bearing(edge.geometry)) > DIR_THRESH:
            continue
        overlap = edge.geometry.intersection(seg_buf).length
        if overlap / max(edge.geometry.length, 1) < OVERLAP_THRESH:
            continue
        if SEVERITY.get(cong, 0) > SEVERITY.get(edge_cong.get(edge_id, 'no data'), 0):
            edge_cong[edge_id] = cong
            n_matched += 1

matched_pct = len(edge_cong) / max(len(edges), 1) * 100
print(f'Edges matched   : {len(edge_cong):,} / {len(edges):,}  ({matched_pct:.1f}%)')
print('\nCongestion distribution:')
from collections import Counter
print(Counter(edge_cong.values()))

---
## Step 6 — Write to DuckDB

In [ ]:
%%time
from datetime import datetime, timezone

with TrafficDB(str(DB_PATH), read_only=False) as db:
    # Ensure congestion_tomtom columns exist
    for col, dtype, default in [
        ('congestion_tomtom',    'VARCHAR',   "'no data'"),
        ('congestion_tomtom_at', 'TIMESTAMP', 'NULL'),
    ]:
        db.con.execute(
            f"ALTER TABLE driving.edges "
            f"ADD COLUMN IF NOT EXISTS {col} {dtype} DEFAULT {default}"
        )

    run_id = db.write_congestion(
        edge_cong,
        source        = 'tomtom',
        zoom          = ZOOM,
        n_segments    = len(traffic),
        boundary_name = NAME,
    )

print(f'Written as run {run_id}  source=tomtom  zoom={ZOOM}')
print(f'congestion_tomtom : {len(edge_cong):,} edges updated')

---
## Step 7 — Compare Mapbox, Google, and TomTom

In [ ]:
%%time
with TrafficDB(str(DB_PATH)) as db:
    print('Mapbox:')
    display(db.get_congestion_summary('mapbox'))
    print('Google:')
    display(db.get_congestion_summary('google'))
    print('TomTom:')
    display(db.con.execute("""
        SELECT congestion_tomtom AS congestion, count(*) AS edges,
               round(sum(length_m)/1000, 1) AS km,
               round(count(*)*100.0 / sum(count(*)) OVER (), 1) AS pct
        FROM driving.edges GROUP BY congestion_tomtom ORDER BY km DESC
    """).df())

In [ ]:
%%time
# Side-by-side: Mapbox vs TomTom
from IPython.display import display, HTML
from html import escape

with TrafficDB(str(DB_PATH)) as db:
    both = db.con.execute("""
        SELECT edge_id, highway, name, length_m,
               congestion_mapbox, congestion_google,
               congestion_tomtom,
               ST_AsText(geometry) AS wkt_geom
        FROM driving.edges
    """).df()

gdf_edges = gpd.GeoDataFrame(
    both.drop(columns=['wkt_geom']),
    geometry=gpd.GeoSeries.from_wkt(both['wkt_geom']),
    crs='EPSG:4326',
)
viz    = gdf_edges[gdf_edges.geometry.geom_type.isin(['LineString','MultiLineString'])].copy()
bds    = viz.total_bounds
center = [(bds[1]+bds[3])/2, (bds[0]+bds[2])/2]

def make_map(col, title):
    m = folium.Map(location=center, zoom_start=13, tiles='OpenStreetMap')
    folium.GeoJson(
        viz.__geo_interface__,
        style_function=lambda feat, c=col: {
            'color':  CONGESTION_COLORS.get(feat['properties'].get(c, 'no data'), '#cccccc'),
            'weight': 3 if feat['properties'].get(c, 'no data') != 'no data' else 1,
        },
        tooltip=folium.GeoJsonTooltip(fields=[col, 'highway', 'name'],
                                      aliases=['Congestion', 'Type', 'Name']),
        popup=None,
    ).add_to(m)
    return escape(m.get_root().render()), title

maps = [
    make_map('congestion_mapbox', 'Mapbox'),
    make_map('congestion_google', 'Google'),
    make_map('congestion_tomtom', 'TomTom'),
]

panels = ''.join(
    f'<div style="flex:1"><h4 style="text-align:center;margin:4px 0">{title}</h4>'
    f'<iframe srcdoc="{html}" width="100%" height="400" frameborder="0"></iframe></div>'
    for html, title in maps
)
display(HTML(f'<div style="display:flex;gap:6px">{panels}</div>'))